# Notebook 2 — Oracle Retrain on Retain Set

**Experiment:** ReGUn benchmark — oracle retrain baseline.

**Prerequisite:** Run **Notebook 1** first and attach its output dataset.
Set `CKPT_DATASET_DIR` below.

This notebook trains a fresh model from scratch on the **RETAIN SET ONLY** per seed,
using the same architecture / hyperparameters / epoch budget as Θ_o from Notebook 1.
This is the gold-standard oracle baseline for unlearning evaluation.

| Stage | Output | Path |
|-------|--------|------|
| **A** | Environment setup, load config | — |
| **B** | Load split files from Notebook 1 | — |
| **C** | Train Θ_retrain per seed | `checkpoints/regun_nb2/retrain/seed{s}.pt` |
| **D** | Evaluate Θ_retrain and save results CSV | `results_oracle_retrain.csv` |

## A. Environment Setup & Load Config

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print('STDERR:', r.stderr[-2000:])
    return r.returncode

sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, copy, random, argparse, collections, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SET THIS to the Kaggle dataset mount path from Notebook 1.
#  Typical: /kaggle/input/<your-regun-nb1-slug>/
# ══════════════════════════════════════════════════════════════════════
CKPT_DATASET_DIR = '/kaggle/input/regun-notebook1'  # ← EDIT THIS

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/regun_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/regun/regun_config.json',
    f'{CKPT_DATASET_DIR}/regun/regun_config.json',
]

config_path = None
CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p
        CKPT_ROOT_NB1 = os.path.dirname(_p)
        break

assert config_path is not None, (
    'regun_config.json not found. Checked:\n' +
    '\n'.join(f'  - {p}' for p in _CONFIG_CANDIDATES) + '\n' +
    'Make sure Notebook 1 finished and the correct dataset is attached.')

with open(config_path) as f:
    CFG = json.load(f)

TEST_MODE         = CFG['TEST_MODE']
TEST_FRACTION     = CFG['TEST_FRACTION']
_MODE_TAG         = CFG['_MODE_TAG']
DATASET           = CFG['DATASET']
ARCH              = CFG['ARCH']
IS_VIT            = CFG['IS_VIT']
NUM_CLASSES       = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES = CFG['CLASS_LABEL_NAMES']
SPLIT_SEEDS       = CFG['SPLIT_SEEDS']
FORGET_FRACTION   = CFG['FORGET_FRACTION']
PRETRAIN_LR       = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS   = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS       = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']
_TOTAL     = {DATASET: CFG['TOTAL']}
_PER_CLASS = {DATASET: CFG['PER_CLASS']}

# Re-root checkpoint paths
_old_root = CFG['CKPT_ROOT']
def _repath(p): return p.replace(_old_root, CKPT_ROOT_NB1)
CKPT_PRETRAIN = _repath(CFG['CKPT_PRETRAIN'])

# Splits directory
SPLIT_DIR = _repath(CFG['SPLIT_DIR'])

for p, name in [(CKPT_PRETRAIN, 'pre_train')]:
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}] {name}: {p}')

DATA_PATH = '/kaggle/working/data'
WORK_ROOT = '/kaggle/working'
# Retrain checkpoints land in writable /kaggle/working/
CKPT_ROOT_NB2 = '/kaggle/working/checkpoints/regun_nb2'
os.makedirs(f'{CKPT_ROOT_NB2}/retrain', exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

print(f'\nMode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Split seeds: {SPLIT_SEEDS}  Forget fraction: {FORGET_FRACTION}')

## B. Load Split Files & Dataset

In [ ]:
from utils import get_dataset, get_model, test, load_encoder_ckpt_safely, SubSet
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=True, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='regun',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _stratified_subset(ds, fraction, seed=42):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]
            rng.shuffle(cls_pool)
            n_keep = max(1, math.ceil(len(cls_pool) * fraction))
            kept.extend(cls_pool[:n_keep])
        sub = torch.utils.data.Subset(ds, kept)
        base_targets = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_targets[i] for i in kept]
        return sub

    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: {TEST_FRACTION*100:.1f}% → '
          f'Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

TEST_KW = dict(batch_size=min(256, len(dataset_test)), num_workers=2,
               pin_memory=True, shuffle=False)
test_loader = torch.utils.data.DataLoader(dataset_test, **TEST_KW)

# ── Load split files ─────────────────────────────────────────────────
splits = {}
for seed in SPLIT_SEEDS:
    split_file = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    if not os.path.exists(split_file):
        raise FileNotFoundError(f'Split file missing: {split_file}\n'
                                f'Make sure Notebook 1 was run and the dataset is attached correctly.')
    with open(split_file) as f:
        splits[seed] = json.load(f)
    n_f = splits[seed]['n_forget']
    n_r = splits[seed]['n_retain']
    print(f'Seed {seed}: forget={n_f}  retain={n_r}  '
          f'forget%={100*n_f/(n_f+n_r):.1f}%')

print(f'All {len(SPLIT_SEEDS)} splits loaded.')

## C. Train Oracle Retrain Model per Seed

Trains a fresh model from scratch on RETAIN SET ONLY per seed,
same architecture / hyperparameters / epoch budget as Θ_o.

In [ ]:
# ── Helper: build SubSet from index list ─────────────────────────────
def make_subset_from_indices(dataset, indices):
    """Build a SubSet-compatible dataset from a list of global indices."""
    return SubSet(dataset, indices)

# ── Helper: evaluate model on arbitrary index subsets ─────────────────
def eval_on_indices(model, dataset, indices, device, batch_size=256):
    """
    Compute accuracy on an arbitrary set of sample indices.
    This is the updated eval harness for the cross-class forget split:
    accuracy is computed over sample INDEX sets, not by class label.
    Returns (accuracy, n_correct, n_total).
    """
    if len(indices) == 0:
        return 0.0, 0, 0
    subset = make_subset_from_indices(dataset, indices)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    acc = correct / max(1, total)
    return acc, correct, total


RETRAIN_RESULTS = []
LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True,
                 shuffle=True, drop_last=True)

for seed in SPLIT_SEEDS:
    split = splits[seed]
    retain_indices = split['retain_indices']
    forget_indices = split['forget_indices']

    print(f'\n{"="*60}')
    print(f'  SEED {seed} — Retrain on retain set ({len(retain_indices)} samples)')
    print(f'{"="*60}')

    ckpt_out = f'{CKPT_ROOT_NB2}/retrain/seed{seed}.pt'
    if os.path.exists(ckpt_out):
        print(f'  Checkpoint exists: {ckpt_out} — skipping training.')
    else:
        # Fresh model from scratch
        args_rt = make_args(
            unlearn_method='retrain',
            epochs_or_steps=PRETRAIN_EPOCHS,
            lr=PRETRAIN_LR,
            patience=PRETRAIN_PATIENCE,
            remove_FC=True, CMFClassifier=True,
        )
        model_rt = get_model(args_rt, device)

        retain_ds  = make_subset_from_indices(dataset_train, retain_indices)
        retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)

        optimizer = optim.SGD(
            model_rt.parameters(), lr=PRETRAIN_LR,
            momentum=0.9, weight_decay=5e-4, nesterov=True
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=PRETRAIN_EPOCHS
        )

        t0 = time.time()
        for epoch in range(1, PRETRAIN_EPOCHS + 1):
            model_rt.train()
            total_loss, total_n = 0.0, 0
            for x, y in retain_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                # CMF: recompute W from retain set each epoch
                if hasattr(model_rt, 'recompute_cmf'):
                    model_rt.eval()
                    model_rt.recompute_cmf(retain_loader, device=device)
                    model_rt.train()
                    loss, _ = model_rt.forward_a((x, y), stage='train')
                else:
                    import torch.nn.functional as F
                    output = model_rt(x)
                    loss = F.cross_entropy(output, y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * x.size(0)
                total_n += x.size(0)
            scheduler.step()
            if epoch % max(1, PRETRAIN_EPOCHS // 5) == 0 or epoch == PRETRAIN_EPOCHS:
                elapsed = time.time() - t0
                print(f'  Epoch {epoch:3d}/{PRETRAIN_EPOCHS}  '
                      f'loss={total_loss/max(1,total_n):.4f}  '
                      f'elapsed={elapsed:.0f}s')

        if hasattr(model_rt, 'recompute_cmf'):
            model_rt.eval()
            model_rt.recompute_cmf(retain_loader, device=device)

        torch.save(model_rt.state_dict(), ckpt_out)
        elapsed = time.time() - t0
        print(f'  → saved: {ckpt_out}  (wall time: {elapsed/60:.1f} min)')

    # ── Evaluate retrain model ─────────────────────────────────────────
    args_ev = make_args(remove_FC=True, CMFClassifier=True)
    model_ev = get_model(args_ev, device)
    model_ev.load_state_dict(torch.load(ckpt_out, map_location=device))

    # Rebuild retain_loader for CMF recompute
    retain_ds = make_subset_from_indices(dataset_train, split['retain_indices'])
    retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
    if hasattr(model_ev, 'recompute_cmf'):
        model_ev.eval()
        model_ev.recompute_cmf(retain_loader, device=device)

    # Output accuracy: test set retain/forget accuracy by index sets
    # Map train forget/retain indices to test set: evaluate on full test set
    # and separately on forget/retain subsets of the TEST set
    # (Using the same approach: eval over index sets not class labels)
    test_all_acc, _, _ = eval_on_indices(
        model_ev, dataset_test,
        list(range(len(dataset_test))), device
    )
    # For the retrain oracle the key metric is test set accuracy
    ra, fa, _ = test(model_ev, device, test_loader, [],
                     CLASS_LABEL_NAMES, NUM_CLASSES, set_name=f'Test (seed {seed})')
    print(f'  Seed {seed}: test_acc={test_all_acc:.4f}  retain_acc={ra:.4f}')

    RETRAIN_RESULTS.append(dict(
        seed=seed,
        n_retain=len(split['retain_indices']),
        n_forget=len(split['forget_indices']),
        test_acc=test_all_acc,
        retain_acc=ra,
        ckpt=ckpt_out,
    ))

print('\nAll oracle retrain runs complete.')

## D. Results Summary & Save CSV

In [ ]:
import pandas as pd

results_df = pd.DataFrame(RETRAIN_RESULTS)
print('\n=== Oracle Retrain Results ===')
print(results_df[['seed', 'n_retain', 'n_forget', 'test_acc', 'retain_acc']].to_string(index=False))
print(f'\nMean test_acc: {results_df["test_acc"].mean():.4f} ± {results_df["test_acc"].std():.4f}')

csv_path = '/kaggle/working/results_oracle_retrain.csv'
results_df.to_csv(csv_path, index=False)
print(f'Results saved: {csv_path}')

# Save NB2 config for downstream notebooks
nb2_config = {
    'CKPT_ROOT_NB2': CKPT_ROOT_NB2,
    'SPLIT_SEEDS': SPLIT_SEEDS,
    'retrain_checkpoints': {s: f'{CKPT_ROOT_NB2}/retrain/seed{s}.pt' for s in SPLIT_SEEDS},
    'results': RETRAIN_RESULTS,
}
with open(f'{CKPT_ROOT_NB2}/nb2_config.json', 'w') as f:
    json.dump(nb2_config, f, indent=2)
print(f'NB2 config saved: {CKPT_ROOT_NB2}/nb2_config.json')